In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

base_results = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/results"
)

paths = {
    "sf1": base_results / "fiben_mongo_sf1" / "schemalens_reduction_analysis_hot.csv",
    "sf10": base_results / "fiben_mongo_sf10" / "schemalens_reduction_analysis_hot.csv",
    "sf30": base_results / "fiben_mongo_sf30" / "schemalens_reduction_analysis_hot.csv",
}

# ============================================================
# LOAD
# ============================================================

dfs = []

for scale, path in paths.items():
    df = pd.read_csv(path)
    df["scale_label"] = scale
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# ============================================================
# 1. SUMMARY (ATUALIZADO)
# ============================================================

summary_df = (
    all_df.groupby("scale_label")
    .agg(
        n_queries=("query_name", "count"),
        avg_DSR=("DSR", "mean"),
        top1_preservation=("top1_preserved_by_activated", "mean"),
        mean_activated_regret=("activated_regret", "mean"),
        mean_primary_regret=("primary_regret", "mean"),
        mean_near_best_ratio=("near_best_ratio", "mean"),

        primary_winners=("best_group", lambda x: (x == "primary").sum()),
        secondary_winners=("best_group", lambda x: (x == "secondary_affected").sum()),
        control_winners=("best_group", lambda x: (x == "control").sum()),
    )
    .reset_index()
)

display(summary_df)

# ============================================================
# 2. P95
# ============================================================

pivot_p95 = all_df.pivot_table(
    index=["official_id", "query_name"],
    columns="scale_label",
    values="best_p95_ms"
).reset_index()

pivot_p95["delta_sf10_vs_sf1"] = (
    pivot_p95["sf10"] - pivot_p95["sf1"]
) / pivot_p95["sf1"]

pivot_p95["delta_sf30_vs_sf1"] = (
    pivot_p95["sf30"] - pivot_p95["sf1"]
) / pivot_p95["sf1"]

display(pivot_p95)

# ============================================================
# 3. STABILITY
# ============================================================

pivot_config = all_df.pivot_table(
    index=["official_id", "query_name"],
    columns="scale_label",
    values="best_config",
    aggfunc="first"
).reset_index()

pivot_config["stable_across_scales"] = pivot_config.apply(
    lambda row: len(set([row["sf1"], row["sf10"], row["sf30"]])) == 1,
    axis=1
)

display(pivot_config)

print("Stable ratio:", pivot_config["stable_across_scales"].mean())

# ============================================================
# 4. NEAR-BEST COMPARISON
# ============================================================

pivot_near = all_df.pivot_table(
    index=["official_id", "query_name"],
    columns="scale_label",
    values="near_best_ratio"
).reset_index()

display(pivot_near)

# ============================================================
# SAVE
# ============================================================

output_dir = base_results / "cross_scale_analysis_fiben"
output_dir.mkdir(exist_ok=True)

summary_df.to_csv(output_dir / "cross_scale_summary.csv", index=False)
pivot_p95.to_csv(output_dir / "cross_scale_p95.csv", index=False)
pivot_config.to_csv(output_dir / "cross_scale_configs.csv", index=False)
pivot_near.to_csv(output_dir / "cross_scale_near_best.csv", index=False)

print("Saved to:", output_dir)

,scale_label,n_queries,avg_DSR,top1_preservation,mean_activated_regret,mean_primary_regret,mean_near_best_ratio,primary_winners,secondary_winners,control_winners
0,sf1,10,0.5,0.8,0.157106,0.256357,0.239881,4,4,2
1,sf10,10,0.5,0.8,0.082722,0.150404,0.338095,7,1,2
2,sf30,10,0.5,0.9,0.000477,0.007773,0.698810,6,3,1


scale_label,official_id,query_name,sf1,sf10,sf30,delta_sf10_vs_sf1,delta_sf30_vs_sf1
0,Q1,Q1_CompanyProfileIBM,0.233373,0.295784,0.224772,0.267430,-0.036857
1,Q10,Q10_CreateAccountHoldingAndBuyTransaction,0.575564,0.427061,0.619045,-0.258012,0.075545
2,Q2,Q2_CompanyWithIndustryCountryAndListedSecurities,0.117117,0.155368,0.250701,0.326600,1.140594
3,Q3,Q3_SecuritiesHeldInEachFinancialServiceAccount,0.434946,0.899143,1.652037,1.067252,2.798256
4,Q4,Q4_CompaniesReachedFromPersonThroughAccountHol...,1.002833,4.308413,17.142359,3.296241,16.093929
5,Q5,Q5_ReportsAndMetricDataOfCompany,37.626791,28.432783,45.217182,-0.244347,0.201728
6,Q6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,0.385924,91.474332,418.834830,236.027027,1084.279030
7,Q7,Q7_PersonsWhoBoughtMoreIBMThanSold,1.186999,0.857698,1.322278,-0.277423,0.113968
8,Q8,Q8_IBMTransactionsBelowAverageSellingPrice,0.863551,0.921652,1.292882,0.067282,0.497169
9,Q9,Q9_PersonsWhoBoughtAndSoldSameStock,0.851476,1.056870,1.223861,0.241220,0.437339


scale_label,official_id,query_name,sf1,sf10,sf30,stable_across_scales
0,Q1,Q1_CompanyProfileIBM,CONTROL,CONTROL,G0,False
1,Q10,Q10_CreateAccountHoldingAndBuyTransaction,G9,G3,G7,False
2,Q2,Q2_CompanyWithIndustryCountryAndListedSecurities,G5,G1,G9,False
3,Q3,Q3_SecuritiesHeldInEachFinancialServiceAccount,G5,G4,G5,False
4,Q4,Q4_CompaniesReachedFromPersonThroughAccountHol...,CONTROL,G5,G4,False
5,Q5,Q5_ReportsAndMetricDataOfCompany,G2,G9,CONTROL,False
6,Q6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,G9,CONTROL,G3,False
7,Q7,Q7_PersonsWhoBoughtMoreIBMThanSold,G4,G6,G3,False
8,Q8,Q8_IBMTransactionsBelowAverageSellingPrice,G7,G5,G5,False
9,Q9,Q9_PersonsWhoBoughtAndSoldSameStock,G7,G5,G7,False


Stable ratio: 0.0


scale_label,official_id,query_name,sf1,sf10,sf30
0,Q1,Q1_CompanyProfileIBM,0.500000,0.500000,0.500000
1,Q10,Q10_CreateAccountHoldingAndBuyTransaction,0.142857,0.285714,0.571429
2,Q2,Q2_CompanyWithIndustryCountryAndListedSecurities,0.166667,0.166667,0.166667
3,Q3,Q3_SecuritiesHeldInEachFinancialServiceAccount,0.285714,0.285714,0.857143
4,Q4,Q4_CompaniesReachedFromPersonThroughAccountHol...,0.142857,0.428571,0.571429
5,Q5,Q5_ReportsAndMetricDataOfCompany,0.250000,0.250000,0.750000
6,Q6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,0.333333,0.166667,1.000000
7,Q7,Q7_PersonsWhoBoughtMoreIBMThanSold,0.125000,0.250000,1.000000
8,Q8,Q8_IBMTransactionsBelowAverageSellingPrice,0.166667,0.333333,1.000000
9,Q9,Q9_PersonsWhoBoughtAndSoldSameStock,0.285714,0.714286,0.571429


Saved to: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/results/cross_scale_analysis_fiben
